# All CNN U Need: train trên Colab

**Lần đầu:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Repo private nên cần token: tạo GitHub token (Settings → Developer settings → Fine-grained tokens, chỉ cấp quyền *Contents: Read* cho repo `all_cnn_u_need`). Sau đó bấm biểu tượng 🔑 bên trái Colab → thêm secret tên `GH_TOKEN` và bật *Notebook access*. Nếu repo public thì bỏ qua bước này.

**Mỗi lần mở:** chạy lần lượt từ trên xuống. Kết quả lưu vào Google Drive `MyDrive/all_cnn_u_need/runs/`, nên Colab có ngắt kết nối cũng không mất. Chạy lại cell 3: model nào train xong rồi sẽ tự bỏ qua.

In [ ]:
# 1. Setup: mount Drive, clone/pull code from GitHub
import os
from google.colab import drive, userdata
drive.mount('/content/drive')

REPO = 'FirstKhoi/all_cnn_u_need'
try:
    token = userdata.get('GH_TOKEN') + '@'  # private repo
except Exception:
    token = ''  # public repo, no token needed

if os.path.exists('/content/all_cnn_u_need'):
    !git -C /content/all_cnn_u_need pull -q
else:
    !git clone -q https://{token}github.com/{REPO}.git /content/all_cnn_u_need
%cd /content/all_cnn_u_need

RUNS = '/content/drive/MyDrive/all_cnn_u_need/runs'  # train.py / visualize.py read RUNS_DIR
os.environ['RUNS_DIR'] = RUNS
!mkdir -p {RUNS} && cp -n runs/* {RUNS}/ 2>/dev/null; ls {RUNS}

In [ ]:
# 2. GPU + model check
!nvidia-smi -L
!python check.py

In [ ]:
# 3. Train everything, cheap -> expensive. Finished runs are skipped, so just re-run this cell after a disconnect.
import json

PLAN = [  # (model, epochs, extra args)
    ('mobilenet_w05', 30, ''),
    ('mobilenet',     30, ''),
    ('vgg11',         30, ''),
    ('vgg16',         30, ''),
    ('plain18',       30, ''),
    ('resnet18',      30, ''),
    ('plain34',       30, ''),
    ('resnet34',      30, ''),
    ('googlenet',     30, ''),
    ('convnext',     100, '--opt adamw'),
]

def done(tag, epochs):
    path = f'{RUNS}/{tag}.json'
    return os.path.exists(path) and len(json.load(open(path))['test_acc']) >= epochs

for model, epochs, extra in PLAN:
    if done(model, epochs):
        print(f'skip {model}: already trained')
        continue
    !python train.py --model {model} --epochs {epochs} --amp {extra}

In [ ]:
# 4. Compare all runs (Mac + Colab)
from IPython.display import Image, display
!python visualize.py compare
!python visualize.py curves
display(Image(f'{RUNS}/compare.png'), Image(f'{RUNS}/curves.png'))

In [ ]:
# 5. Feed one test image through every trained model: what each conv layer "sees" (8 strongest channels per layer)
IDX = 3  # test image index, 3 = plane
for model, _, _ in PLAN:
    !python visualize.py features {model} --idx {IDX}
    display(Image(f'{RUNS}/{model}_features.png'))